In [13]:
import numpy as np
import pandas as pd

In [14]:
# SETTINGS
CSV_PATH = "../data/vessel_prediction_model_data/mid_model_data/mid_test_data.csv"   # change if needed
EARTH_RADIUS_M = 6371000.0
KNOT_TO_MPS = 0.514444
YARDS_PER_METER = 1.09361

In [15]:
# DEAD RECKONING FUNCTION
def dead_reckoning_next_point(lat_deg, lon_deg, speed_knots, cog_sin, cog_cos, dt_seconds):
    """
    Predict next point assuming constant speed and constant course.
    Uses cog_sin and cog_cos directly from the dataset.
    Returns predicted (lat, lon) in degrees.
    """
    dist_m = speed_knots * KNOT_TO_MPS * dt_seconds
 
    # normalize direction vector in case stored sin/cos are slightly imperfect
    norm = np.sqrt(cog_sin**2 + cog_cos**2)
    if norm <= 1e-12 or np.isnan(norm):
        return np.nan, np.nan
 
    cog_sin = cog_sin / norm
    cog_cos = cog_cos / norm
 
    d_north = dist_m * cog_cos
    d_east = dist_m * cog_sin
 
    dlat = d_north / 111320.0
 
    cos_lat = np.cos(np.radians(lat_deg))
    if np.abs(cos_lat) < 1e-10:
        return np.nan, np.nan
 
    dlon = d_east / (111320.0 * cos_lat)
 
    pred_lat = lat_deg + dlat
    pred_lon = lon_deg + dlon
    return pred_lat, pred_lon

In [16]:
# HAVERSINE ERROR
def haversine_m(lat1, lon1, lat2, lon2):
    """
    Great-circle distance in meters.
    Inputs can be scalars or numpy arrays.
    """
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
 
    dlat = lat2 - lat1
    dlon = lon2 - lon1
 
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_M * c

In [17]:
# LOAD DATA
df = pd.read_csv(CSV_PATH)
 
required_cols = [
    "MMSI",
    "voyage_id",
    "TIME",
    "LAT",
    "LON",
    "SPEED",
    "dt",
    "cog_sin",
    "cog_cos",
]

In [18]:
# BUILD TRUE NEXT POINTS WITHIN SAME MMSI/Voyage
group_cols = ["MMSI", "voyage_id"]
 
df["next_LAT"] = df.groupby(group_cols)["LAT"].shift(-1)
df["next_LON"] = df.groupby(group_cols)["LON"].shift(-1)
df["next_TIME"] = df.groupby(group_cols)["TIME"].shift(-1)
 
pairs = df.dropna(subset=["next_LAT", "next_LON", "next_TIME"]).copy()

In [19]:
# RUN DEAD RECKONING
pred_lats = np.zeros(len(pairs), dtype=np.float64)
pred_lons = np.zeros(len(pairs), dtype=np.float64)
 
for i, row in enumerate(pairs.itertuples(index=False)):
    pred_lat, pred_lon = dead_reckoning_next_point(
        lat_deg=row.LAT,
        lon_deg=row.LON,
        speed_knots=row.SPEED,
        cog_sin=row.cog_sin,
        cog_cos=row.cog_cos,
        dt_seconds=row.dt,
    )
    pred_lats[i] = pred_lat
    pred_lons[i] = pred_lon
 
pairs["dr_pred_lat"] = pred_lats
pairs["dr_pred_lon"] = pred_lons

In [20]:
# COMPUTE ERROR
pairs["error_m"] = haversine_m(
    pairs["dr_pred_lat"].values,
    pairs["dr_pred_lon"].values,
    pairs["next_LAT"].values,
    pairs["next_LON"].values,
)
 
pairs["error_yds"] = pairs["error_m"] * YARDS_PER_METER

In [21]:
# SUMMARY METRICS
mse_m = np.mean(pairs["error_yds"].values ** 2)
rmse_m = np.sqrt(mse_m)
mae_m = np.mean(np.abs(pairs["error_yds"].values))
median_m = np.median(pairs["error_yds"].values)
p95_m = np.percentile(pairs["error_yds"].values, 95)
 
q1_m = np.percentile(pairs["error_yds"].values, 25)
q3_m = np.percentile(pairs["error_yds"].values, 75)
iqr_m = q3_m - q1_m
upper_bound_m = q3_m + 1.5 * iqr_m
outlier_mask = pairs["error_yds"].values > upper_bound_m
outlier_count = int(np.sum(outlier_mask))
outlier_percent = 100.0 * outlier_count / len(pairs)
 
summary = pd.DataFrame([{
    "n_predictions": len(pairs),
    "rmse_m": rmse_m,
    "mae_m": mae_m,
    "median_m": median_m,
    "p95_m": p95_m,
    "q1_m": q1_m,
    "q3_m": q3_m,
    "iqr_m": iqr_m,
    "upper_bound_m": upper_bound_m,
    "outlier_count": outlier_count,
    "outlier_percent": outlier_percent,
    "rmse_yds": rmse_m * YARDS_PER_METER,
    "mae_yds": mae_m * YARDS_PER_METER,
    "median_yds": median_m * YARDS_PER_METER,
    "p95_yds": p95_m * YARDS_PER_METER,
    "q1_yds": q1_m * YARDS_PER_METER,
    "q3_yds": q3_m * YARDS_PER_METER,
    "iqr_yds": iqr_m * YARDS_PER_METER,
    "upper_bound_yds": upper_bound_m * YARDS_PER_METER,
    "mse_m": mse_m,
    "mse_yds": mse_m * (YARDS_PER_METER ** 2),
}])

In [23]:
# INTERPRETABLE PRINT OUTPUT
print("\n" + "=" * 70)
print("DEAD RECKONING BASELINE SUMMARY")
print("=" * 70)
 
print(f"Total valid predictions: {len(pairs):,}")
 
print("\nTypical error:")
print(f"  MAE:        {mae_m:,.2f} yds")
print(f"  Median:     {median_m:,.2f} yds")
 
print("\nSpread / consistency:")
print(f"  RMSE:       {rmse_m:,.2f} yds")
print(f"  Q1:         {q1_m:,.2f} yds")
print(f"  Q3:         {q3_m:,.2f} yds")
print(f"  IQR:        {iqr_m:,.2f} yds")
 
print("\nWorst-case behavior:")
print(f"  95th pct:   {p95_m:,.2f} yds")
print(f"  Outlier cutoff (boxplot rule): {upper_bound_m:,.2f} yds")
print(f"  Outliers:   {outlier_count:,} / {len(pairs):,}  ({outlier_percent:.2f}%)")
 
print("\nRaw squared-error metrics:")
print(f"  MSE:        {mse_m:,.2f} yds^2")
print(f"  RMSE:       {rmse_m:,.2f} yds")
 
print("=" * 70)
 
print("\nTOP 10 WORST DEAD RECKONING PREDICTIONS")
print("-" * 70)
 
worst_10 = pairs.nlargest(10, "error_yds")[
    [
        "MMSI", "voyage_id", "TIME", "LAT", "LON", "SPEED", "dt",
        "cog_sin", "cog_cos",
        "next_LAT", "next_LON",
        "dr_pred_lat", "dr_pred_lon",
        "error_yds"
    ]
].copy()

worst_10["error_yds"] = worst_10["error_yds"].round(2)
 
print(worst_10.to_string(index=False))


DEAD RECKONING BASELINE SUMMARY
Total valid predictions: 3,164

Typical error:
  MAE:        2,286.99 yds
  Median:     817.19 yds

Spread / consistency:
  RMSE:       3,884.38 yds
  Q1:         189.04 yds
  Q3:         3,166.12 yds
  IQR:        2,977.08 yds

Worst-case behavior:
  95th pct:   9,686.15 yds
  Outlier cutoff (boxplot rule): 7,631.75 yds
  Outliers:   269 / 3,164  (8.50%)

Raw squared-error metrics:
  MSE:        15,088,434.54 yds^2
  RMSE:       3,884.38 yds

TOP 10 WORST DEAD RECKONING PREDICTIONS
----------------------------------------------------------------------
     MMSI    voyage_id                TIME       LAT        LON  SPEED     dt   cog_sin   cog_cos  next_LAT   next_LON  dr_pred_lat  dr_pred_lon  error_yds
209425000 12_209425000 2024-06-19 20:15:00 21.870388 -77.090018   14.4  300.0 -0.763796  0.645458 21.783791 -76.950278    21.883274   -77.106449   21380.42
209425000  1_209425000 2023-12-06 21:00:00 23.235393 -78.993713   12.7  600.0  0.322266 -0.94664